# Anima Heatmap · ComfyUI 실행

GPU 런타임에서 순서대로 실행합니다. 이 노트북은 소스·모델 설치, 서버 준비 확인, 터널 실행만 담당합니다.

기본 설정으로 GitHub의 HisameOgasahara/Anima_HEATMAP 저장소를 clone합니다. 별도 ZIP 업로드 없이 순서대로 실행하세요.

Quick Tunnel 주소를 아는 사람은 ComfyUI에 접속할 수 있습니다. 개인 실험용이며 사용 후 마지막 셀로 서버와 터널을 종료하세요.


In [ ]:
# 1. 설정
from pathlib import Path
import sys, subprocess, urllib.request, json, shutil, zipfile, tarfile, os

WORK = Path('/content')
COMFY = WORK / 'ComfyUI'
COMFY_REPO = 'https://github.com/comfyanonymous/ComfyUI.git'
COMFY_REF = 'master'  # 재현이 필요하면 사용할 commit 또는 tag를 지정
SOURCE_REPO = 'https://github.com/HisameOgasahara/Anima_HEATMAP.git'  # 비우면 프로젝트 ZIP 업로드
SOURCE_REF = 'main'
SOURCE_SUBDIR = '.'  # 저장소 루트에 pyproject.toml이 있음
PORT = 8188
MODEL_REPO = 'circlestone-labs/Anima'
MODEL_REVISION = 'main'
MODEL_FILES = {
    'diffusion_models': 'anima-base-v1.0.safetensors',
    'text_encoders': 'qwen_3_06b_base.safetensors',
    'vae': 'qwen_image_vae.safetensors',
}

def run(*args, cwd=None):
    subprocess.run([str(x) for x in args], cwd=cwd, check=True)

def download(url, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file():
        return target
    partial = target.with_suffix(target.suffix + '.partial')
    with urllib.request.urlopen(url, timeout=60) as source, partial.open('wb') as output:
        shutil.copyfileobj(source, output)
    partial.replace(target)
    return target

server = globals().get('server')
tunnel = globals().get('tunnel')


In [ ]:
# 2. ComfyUI와 분석 모듈 소스 준비
if not COMFY.exists():
    run('git', 'clone', COMFY_REPO, COMFY)
run('git', 'fetch', 'origin', COMFY_REF, cwd=COMFY)
run('git', 'checkout', '--detach', 'FETCH_HEAD', cwd=COMFY)

if SOURCE_REPO:
    checkout = WORK / 'anima-heatmap-source'
    if not checkout.exists():
        run('git', 'clone', SOURCE_REPO, checkout)
    run('git', 'fetch', 'origin', SOURCE_REF, cwd=checkout)
    run('git', 'checkout', '--detach', 'FETCH_HEAD', cwd=checkout)
    SOURCE = (checkout / SOURCE_SUBDIR).resolve()
else:
    from google.colab import files
    import uuid
    uploaded = files.upload()
    if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith('.zip'):
        raise ValueError('Anima_Heatmap 프로젝트 ZIP 하나를 업로드하세요.')
    archive = Path(next(iter(uploaded)))
    extracted = WORK / ('anima-source-' + uuid.uuid4().hex)
    extracted.mkdir()
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            target = (extracted / member.filename).resolve()
            if not target.is_relative_to(extracted.resolve()):
                raise ValueError('ZIP에 허용되지 않는 경로가 있습니다.')
        bundle.extractall(extracted)
    candidates = [p.parent for p in extracted.rglob('pyproject.toml') if (p.parent / 'anima_heatmap').is_dir()]
    if len(candidates) != 1:
        raise ValueError('ZIP에서 Anima Heatmap 프로젝트를 하나로 식별할 수 없습니다.')
    SOURCE = candidates[0]
if not (SOURCE / 'anima_heatmap').is_dir():
    raise FileNotFoundError(f'분석 모듈을 찾을 수 없습니다: {SOURCE}')
print('모듈 소스:', SOURCE)


In [ ]:
# 3. 전용 Python 환경과 커스텀 노드 설치
UV_ROOT = WORK / 'anima-uv'
UV_ROOT.mkdir(exist_ok=True)
UV = UV_ROOT / 'uv-x86_64-unknown-linux-gnu' / 'uv'
if not UV.exists():
    archive = download('https://github.com/astral-sh/uv/releases/latest/download/uv-x86_64-unknown-linux-gnu.tar.gz', UV_ROOT / 'uv.tar.gz')
    with tarfile.open(archive) as bundle:
        bundle.extractall(UV_ROOT, filter='data')
ENV = WORK / 'anima-runtime'
PYTHON = ENV / 'bin/python'
if not PYTHON.exists():
    run(UV, 'venv', '--system-site-packages', '--python', sys.executable, ENV)
run(UV, 'pip', 'install', '--python', PYTHON, '-r', COMFY / 'requirements.txt', '-e', SOURCE)
node_target = COMFY / 'custom_nodes/comfyui_anima_heatmap'
if node_target.is_symlink():
    if node_target.resolve() != (SOURCE / 'comfyui_anima_heatmap').resolve():
        node_target.unlink()
        node_target.symlink_to(SOURCE / 'comfyui_anima_heatmap', target_is_directory=True)
elif node_target.exists():
    raise FileExistsError(f'기존 노드 폴더를 확인하세요: {node_target}')
else:
    node_target.symlink_to(SOURCE / 'comfyui_anima_heatmap', target_is_directory=True)
run(PYTHON, '-c', 'import torch, anima_heatmap; print("CUDA:", torch.cuda.is_available()); print("모듈:", anima_heatmap.__file__); assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."')


In [ ]:
# 4. 모델 다운로드
from urllib.parse import quote
for folder, filename in MODEL_FILES.items():
    print('모델 준비:', filename, flush=True)
    url = f'https://huggingface.co/{MODEL_REPO}/resolve/{quote(MODEL_REVISION, safe="")}/split_files/{folder}/{quote(filename)}'
    download(url, COMFY / 'models' / folder / filename)
print('모델 준비 완료')


In [ ]:
# 5. ComfyUI 실행 → 서버 응답·노드 등록 확인
import importlib.util
spec = importlib.util.spec_from_file_location('anima_notebook_runtime', SOURCE / 'notebooks/runtime.py')
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)
if server is not None and server.poll() is None:
    raise RuntimeError('기존 서버를 마지막 셀에서 종료한 뒤 다시 실행하세요.')
required = ['AnimaHeatmapTextEncode', 'AnimaHeatmapSettings', 'AnimaHeatmapSampler', 'AnimaHeatmapView', 'AnimaHeatmapLoadSession']
server, local_url, server_log = runtime.start_server(COMFY, PYTHON, PORT, required_nodes=required)
print('서버와 커스텀 노드 준비 완료:', local_url)
print('서버 로그:', server_log)


In [ ]:
# 6. 준비된 서버에 터널 연결 → 외부 응답 확인 → 주소 표시
if server is None or server.poll() is not None:
    raise RuntimeError('5번 셀에서 서버를 먼저 실행하세요.')
if tunnel is not None and tunnel.poll() is None:
    raise RuntimeError('기존 터널을 마지막 셀에서 종료하세요.')
CLOUDFLARED = download('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', WORK / 'cloudflared')
CLOUDFLARED.chmod(0o755)
tunnel, public_url = runtime.start_tunnel(CLOUDFLARED, local_url, WORK / 'anima_tunnel.log', server)
from IPython.display import display, Markdown
display(Markdown(f'[ComfyUI 열기]({public_url})'))
print('분석 결과 폴더:', COMFY / 'output/anima_heatmap')


## ComfyUI 연결

1. `UNETLoader`에서 Anima, `CLIPLoader`의 anima 타입에서 Qwen, `VAELoader`에서 Qwen Image VAE를 선택합니다.
2. 긍정·부정 프롬프트를 각각 `Anima Heatmap · Text Encode`로 인코딩합니다.
3. `Anima Heatmap · Settings`와 모델·조건·latent를 `Anima Heatmap · KSampler`에 연결합니다.
4. sampler의 latent를 VAE Decode로 보낸 뒤, 이미지와 sampler의 session을 `Anima Heatmap · View`에 연결합니다.
5. View의 단어·단계·layer를 바꿔 다시 실행하면 저장한 지도로 다시 분석합니다.

서버/터널 로그는 `/content/ComfyUI/anima_heatmap_server.log`, `/content/anima_tunnel.log`에 있습니다.


In [ ]:
# 7. 사용 종료
if 'runtime' in globals():
    runtime.stop_process(globals().get('tunnel'))
    runtime.stop_process(globals().get('server'))
server = tunnel = None
print('서버와 터널을 종료했습니다. 저장된 파일은 유지됩니다.')
